# 01 · CICIDS2017: load, inspect and clean

Day 1. Loads the eight MachineLearningCVE CSVs, shows the known problems
(whitespace in column names, a duplicated column, `inf`/`NaN`, broken Web Attack
labels, duplicate rows, constant columns), runs `ids.data.clean`, and saves
the cleaned parquet.

Get the data first: see `data/README.md`.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Works whether Jupyter starts in the repo root or in notebooks/.
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))  # not needed after `pip install -e .`

from ids.data.clean import LABEL_COL, clean, duplicate_columns, load_raw

RAW_DIR = Path(os.environ.get("IDS_RAW_DIR", ROOT / "data/raw/MachineLearningCVE"))
OUT_DIR = ROOT / "data/processed"
BENIGN_FRAC = 0.2  # keep 20% of benign flows so training stays fast

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
sorted(p.name for p in RAW_DIR.glob("*.csv"))

## 1. Load

One CSV per capture day. Expect about 2.83M rows and 79 columns in total.

In [ ]:
raw = load_raw(RAW_DIR)
print(f"{len(raw):,} rows x {raw.shape[1]} columns, "
      f"{raw.memory_usage(deep=True).sum() / 1e9:.2f} GB in memory")
raw.head()

## 2. What is wrong with the raw data

**Column names.** The raw headers start with spaces (`' Destination Port'`); `load_raw` strips them. One header, `Fwd Header Length`, appears twice, so pandas names the copy `Fwd Header Length.1`.

In [ ]:
first_csv = sorted(RAW_DIR.glob("*.csv"))[0]
header = pd.read_csv(first_csv, nrows=0, encoding="latin-1").columns
print("names with leading/trailing spaces:", sum(c != c.strip() for c in header))
print("columns that duplicate another:", duplicate_columns(raw.drop(columns=LABEL_COL)))

**Infinite and missing values.** These come from divisions by a zero flow duration, mostly in `Flow Bytes/s` and `Flow Packets/s`.

In [ ]:
num = raw.select_dtypes("number")
bad = pd.DataFrame({
    "inf": np.isinf(num).sum(),
    "nan": num.isna().sum(),
}).query("inf > 0 or nan > 0")
bad

**Labels.** The Web Attack labels contain an en dash that decodes as a garbage character. Note the `repr` below.

In [ ]:
raw[LABEL_COL].value_counts(dropna=False).rename_axis("raw label").to_frame().assign(
    repr=lambda d: [repr(x) for x in d.index]
)

**Duplicates and constant columns.**

In [ ]:
print(f"exact duplicate rows: {raw.duplicated().sum():,}")
consts = [c for c in num.columns if num[c].nunique(dropna=False) <= 1]
print("constant columns:", consts)

## 3. Clean

Runs every step in `src/ids/data/clean.py` and reports what each one removed. Benign rows are downsampled to `BENIGN_FRAC` at the end; attack rows are all kept.

In [ ]:
df, summary = clean(raw, benign_frac=BENIGN_FRAC)
del raw, num  # free memory
{k: v for k, v in summary.items() if k != "label_counts"}

In [ ]:
print(f"{len(df):,} rows x {df.shape[1]} columns")
assert np.isfinite(df.select_dtypes("number").to_numpy(np.float64)).all()
assert not df.duplicated().any()
df[LABEL_COL].value_counts().to_frame("rows")

## 4. A first look at the classes

Heavily imbalanced even after downsampling benign. Infiltration and Heartbleed have only a handful of rows; day 2 drops them.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
counts = df[LABEL_COL].value_counts()
sns.barplot(x=counts.values, y=counts.index, ax=axes[0], color="steelblue")
axes[0].set(xscale="log", xlabel="rows (log scale)", ylabel="", title="Rows per label")
fam = df["Family"].value_counts()
sns.barplot(x=fam.values, y=fam.index, ax=axes[1], color="indianred")
axes[1].set(xscale="log", xlabel="rows (log scale)", ylabel="", title="Rows per family")
plt.tight_layout()
(ROOT / "reports/figures").mkdir(parents=True, exist_ok=True)
fig.savefig(ROOT / "reports/figures/01_class_counts.png", dpi=120)

## 5. Save

In [ ]:
import json

OUT_DIR.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUT_DIR / "cicids2017_clean.parquet", index=False)
(OUT_DIR / "cleaning_summary.json").write_text(json.dumps(summary, indent=2))
print("saved", OUT_DIR / "cicids2017_clean.parquet",
      f"({(OUT_DIR / 'cicids2017_clean.parquet').stat().st_size / 1e6:.0f} MB)")

## Notes for day 2

- Negative values exist in some columns (e.g. `Init_Win_bytes_forward = -1` means "not seen"). Keep them, but check `Flow Duration < 0`.
- Drop leaky features before modelling: `Destination Port` at least.
- Next notebook: `02_preprocessing.ipynb` (families, feature selection, stratified split).